# 11: Final performance summary

This notebook consolidates the frozen 26-feature results. It reads saved artifacts only; it does not fit models, select features, tune parameters, or change decisions.


## 1. Setup

Locate the project and verify that all required final-run outputs exist.


In [1]:
from pathlib import Path
import io
import json

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_root = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
sensitivity_root = project_root / "results/final_pipeline/07_sensitivity_analyses/08_sensitivity_analyses"
interpretation_root = project_root / "results/final_pipeline/08_interpretation_and_reporting/10_model_interpretation"
output_directory = final_root / "11_final_performance_summary"
output_directory.mkdir(parents=True, exist_ok=True)

required = {
    "performance": final_root / "07_tuning_and_outer_evaluation/system_summary.csv",
    "paired": final_root / "07_tuning_and_outer_evaluation/paired_corrected_uncertainty.csv",
    "selected": final_root / "07_tuning_and_outer_evaluation/selected_pipelines.csv",
    "statistics": final_root / "04_statistical_feature_analysis/feature_statistical_summary.csv",
    "sensitivities": sensitivity_root / "sensitivity_system_summary.csv",
    "sensitivity_differences": sensitivity_root / "sensitivity_minus_primary_uncertainty.csv",
    "models": final_root / "09_full_cohort_fit/models/model_manifest.json",
    "full_selected": final_root / "09_full_cohort_fit/full_cohort_selected_pipelines.csv",
    "interpretation": interpretation_root / "permutation_importance_summary.csv",
}
missing = [name for name, path in required.items() if not path.is_file()]
assert not missing, f"Missing required inputs: {missing}"
print("Output:", output_directory.relative_to(project_root))


Output: results/final_pipeline/06_final_fit_and_performance_summary/11_final_performance_summary


## 2. Upstream validation

Confirm that the notebooks supplying this summary passed their saved checks.


In [2]:
validation_paths = {
    "aggregation": final_root / "02_patient_level_aggregation/aggregation_validation.csv",
    "splits": final_root / "03_setup_and_splits/split_validation.csv",
    "statistics": final_root / "04_statistical_feature_analysis/statistical_validation.csv",
    "feature screening": final_root / "05_feature_pipeline_screening/feature_pipeline_screening_validation.csv",
    "family screening": final_root / "06_model_family_screening/model_family_screening_validation.csv",
    "tuning": final_root / "07_tuning_and_outer_evaluation/focused_tuning_validation.csv",
    "outer evaluation": final_root / "07_tuning_and_outer_evaluation/outer_evaluation_validation.csv",
    "sensitivities": sensitivity_root / "sensitivity_validation.csv",
    "full fit": final_root / "09_full_cohort_fit/full_cohort_validation.csv",
    "interpretation": interpretation_root / "interpretation_validation.csv",
}

validation_rows = []
for step, path in validation_paths.items():
    frame = pd.read_csv(path)
    result_column = "passed" if "passed" in frame else "result"
    passed = frame[result_column].astype(str).str.lower().isin(["true", "1", "yes"])
    validation_rows.append({"step": step, "passed": bool(passed.all()), "checks": len(frame)})

upstream_validation = pd.DataFrame(validation_rows)
display(upstream_validation)
assert upstream_validation["passed"].all()


,step,passed,checks
0,aggregation,True,17
1,splits,True,15
2,statistics,True,13
3,feature screening,True,15
4,family screening,True,11
5,tuning,True,9
6,outer evaluation,True,13
7,sensitivities,True,14
8,full fit,True,13
9,interpretation,True,6


## 3. Primary performance

Show the mean performance of the direct and two-stage systems across the 20 paired outer splits.


In [3]:
performance = pd.read_csv(required["performance"])
performance_table = performance[[
    "system", "mean_macro_f1", "mean_balanced_accuracy", "mean_accuracy",
    "mean_macro_auc_ovr", "mean_recall_no_fall", "mean_recall_rare_fall",
    "mean_recall_recurrent_fall",
]].copy()
display(performance_table.round(3))


,system,mean_macro_f1,mean_balanced_accuracy,mean_accuracy,mean_macro_auc_ovr,mean_recall_no_fall,mean_recall_rare_fall,mean_recall_recurrent_fall
0,direct,0.513,0.550,0.624,0.753,0.710,0.384,0.557
1,two_stage,0.510,0.538,0.636,0.747,0.743,0.351,0.522


### Paired comparison

Report direct-minus-two-stage differences with the saved corrected intervals.


In [4]:
paired = pd.read_csv(required["paired"])
paired_table = paired[[
    "metric", "mean_direct_minus_two_stage", "ci95_low", "ci95_high",
    "corrected_resampled_p_value",
]].copy()
display(paired_table.round(3))


,metric,mean_direct_minus_two_stage,ci95_low,ci95_high,corrected_resampled_p_value
0,macro_f1,0.003,-0.034,0.041,0.850
1,balanced_accuracy,0.012,-0.045,0.069,0.669
2,accuracy,-0.012,-0.038,0.015,0.362
3,recall_no_fall,-0.033,-0.090,0.024,0.243
4,recall_rare_fall,0.033,-0.053,0.119,0.430
5,recall_recurrent_fall,0.035,-0.171,0.241,0.726


## 4. Sensitivities

Compare each one-change branch with the unchanged primary procedure.


In [5]:
sensitivity_scores = pd.read_csv(required["sensitivities"])
sensitivity_differences = pd.read_csv(required["sensitivity_differences"])
macro_differences = sensitivity_differences.loc[
    sensitivity_differences["metric"].eq("macro_f1"),
    ["sensitivity_id", "system", "mean_difference", "ci95_low", "ci95_high"],
]
sensitivity_table = sensitivity_scores.merge(
    macro_differences,
    on=["sensitivity_id", "system"],
    how="left",
    validate="many_to_one",
)
display(sensitivity_table[[
    "sensitivity_id", "system", "mean_macro_f1", "mean_recall_rare_fall",
    "mean_difference", "ci95_low", "ci95_high",
]].round(3))


,sensitivity_id,system,mean_macro_f1,mean_recall_rare_fall,mean_difference,ci95_low,ci95_high
0,PRIMARY_NOTEBOOK_07,direct,0.513,0.384,NaN,NaN,NaN
1,PRIMARY_NOTEBOOK_07,two_stage,0.510,0.351,NaN,NaN,NaN
2,S01_101_FLAGS,direct,0.514,0.385,0.001,-0.017,0.019
3,S01_101_FLAGS,two_stage,0.508,0.357,-0.002,-0.030,0.026
4,S02_ON_OFF,direct,0.506,0.370,-0.007,-0.039,0.025
5,S02_ON_OFF,two_stage,0.503,0.343,-0.007,-0.032,0.018
6,S03_COHORT_1043,direct,0.518,0.388,0.005,-0.025,0.035
7,S03_COHORT_1043,two_stage,0.510,0.357,-0.000,-0.021,0.021
8,S04_REVISION_PREPROCESSING,direct,0.522,0.390,0.008,-0.021,0.038
9,S04_REVISION_PREPROCESSING,two_stage,0.517,0.362,0.007,-0.019,0.034


## 5. Statistical feature results

Summarize training-partition FDR stability and the largest univariate effects.


In [6]:
statistics = pd.read_csv(required["statistics"])
statistical_table = statistics[[
    "feature", "descriptive_name", "fdr_significant_partitions",
    "common_test_median_effect", "effect_size_metric",
]].sort_values(
    ["fdr_significant_partitions", "common_test_median_effect"],
    ascending=[False, False],
).reset_index(drop=True)

stability_counts = pd.DataFrame({
    "stability": ["100/100 partitions", "1-99/100 partitions", "0/100 partitions"],
    "features": [
        int(statistics["fdr_significant_partitions"].eq(100).sum()),
        int(statistics["fdr_significant_partitions"].between(1, 99).sum()),
        int(statistics["fdr_significant_partitions"].eq(0).sum()),
    ],
})
display(stability_counts)
display(statistical_table.head(10).round(3))


,stability,features
0,100/100 partitions,16
1,1-99/100 partitions,8
2,0/100 partitions,2


,feature,descriptive_name,fdr_significant_partitions,common_test_median_effect,effect_size_metric
0,NQ_GAUSSIAN_REVISION,Neuro-QoL Gaussian mobility score (8 items),100,0.156,epsilon_squared
1,FRZGT12M,Freezing of gait severity,100,0.153,epsilon_squared
2,NHY_COMBINED_MAX,"Hoehn–Yahr stage, combined states",100,0.150,epsilon_squared
3,NP3GAIT_COMBINED_MAX,"Part III gait, combined states",100,0.132,epsilon_squared
4,NP3PSTBL_COMBINED_MAX,"Part III postural stability, combined states",100,0.113,epsilon_squared
5,Years_since_PD_diagnosis,Years since PD diagnosis,100,0.109,epsilon_squared
6,NP4TOT,MDS-UPDRS Part IV complications total,100,0.109,epsilon_squared
7,NP1RTOT,MDS-UPDRS Part I rater total,100,0.103,epsilon_squared
8,NP3TOT_COMBINED_MAX,"MDS-UPDRS Part III motor total, combined states",100,0.092,epsilon_squared
9,NP1CNST,Constipation severity,100,0.091,epsilon_squared


## 6. Selected procedures

Separate feature-pipeline choices from prediction-model families, then show the three full-cohort artifacts.


In [7]:
selected = pd.read_csv(required["selected"])
selector_labels = {
    "none": "All candidates",
    "corrected_fdr": "FDR statistical screening",
    "l1": "L1 sparse selection",
    "extra_trees": "Tree-importance selection",
}
selected["feature_pipeline"] = selected.apply(
    lambda row: (
        f"Engineered: {row['branch']}"
        if row["candidate_kind"] == "engineered_representation"
        else selector_labels[row["selector"]]
    ),
    axis=1,
)
pipeline_frequency = (
    selected.groupby(["target", "feature_pipeline"], as_index=False)
    .size().rename(columns={"size": "selected_splits"})
    .sort_values(["target", "selected_splits", "feature_pipeline"], ascending=[True, False, True])
)
model_frequency = (
    selected.groupby(["target", "model_family"], as_index=False)
    .size().rename(columns={"size": "selected_splits"})
    .sort_values(["target", "selected_splits", "model_family"], ascending=[True, False, True])
)
display(pipeline_frequency)
display(model_frequency)


,target,feature_pipeline,selected_splits
5,direct,FDR statistical screening,7
6,direct,Tree-importance selection,7
1,direct,Engineered: INT_GAIT_X_POSTURAL_STABILITY,2
0,direct,Engineered: INT_DURATION_X_MOTOR,1
2,direct,Engineered: INT_MOTOR_X_COGNITION,1
3,direct,Engineered: PCA_PART_I_PATIENT_3_80,1
4,direct,Engineered: PCA_PART_I_PATIENT_3_90,1
14,stage_1,L1 sparse selection,7
15,stage_1,Tree-importance selection,4
12,stage_1,Engineered: PCA_PART_I_PATIENT_3_90,2


,target,model_family,selected_splits
3,direct,random_forest,9
0,direct,catboost,7
4,direct,rbf_svc,2
1,direct,extra_trees,1
2,direct,linear_svc,1
6,stage_1,extra_trees,6
9,stage_1,rbf_svc,5
7,stage_1,logistic,4
8,stage_1,random_forest,3
5,stage_1,catboost,2


### Full-cohort artifacts

These fitted models support interpretation and future use; they are not additional performance estimates.


In [8]:
model_manifest = json.loads(required["models"].read_text())
full_selected = pd.read_csv(required["full_selected"]).set_index("target")
model_rows = []
for target, entry in model_manifest["models"].items():
    row = full_selected.loc[target]
    model_rows.append({
        "target": target,
        "feature_selection": row["pipeline_key"],
        "model_family": entry["model_family"],
        "selected_source_groups": int(row["selected_group_count"]),
        "training_patients": int(entry["training_patients"]),
        "parameters": json.dumps(entry["parameters"], sort_keys=True),
    })
full_model_table = pd.DataFrame(model_rows)
display(full_model_table)


,target,feature_selection,model_family,selected_source_groups,training_patients,parameters
0,direct,raw__corrected_fdr__q<=0.05,catboost,20,1040,"{""auto_class_weights"": ""Balanced"", ""depth"": 6,..."
1,stage_1,raw__corrected_fdr__q<=0.05,extra_trees,20,1040,"{""class_weight"": ""balanced"", ""max_depth"": null..."
2,stage_2,raw__extra_trees__threshold=median,random_forest,13,328,"{""class_weight"": null, ""max_depth"": null, ""min..."


## 7. Interpretation

Show the leading grouped permutation results for both final systems.


In [9]:
interpretation = pd.read_csv(required["interpretation"])
interpretation_table = (
    interpretation.loc[
        interpretation["model"].isin(["direct_system", "two_stage_system"])
        & interpretation["metric"].eq("macro_f1"),
        ["model", "label", "mean_drop", "sd_drop", "folds_positive"],
    ]
    .sort_values(["model", "mean_drop"], ascending=[True, False])
    .groupby("model", as_index=False)
    .head(10)
    .reset_index(drop=True)
)
display(interpretation_table.round(4))

,model,label,mean_drop,sd_drop,folds_positive
0,direct_system,Constipation,0.0209,0.0165,4
1,direct_system,"Freezing of gait, preceding 12 months",0.0206,0.0177,4
2,direct_system,Neuro-QoL Gaussian mobility score (8 items),0.0189,0.0083,5
3,direct_system,Age,0.0134,0.0082,5
4,direct_system,MDS-UPDRS Part III motor total,0.0087,0.0087,5
5,direct_system,Urinary problems,0.0063,0.0078,4
6,direct_system,Daytime sleepiness,0.0062,0.0037,4
7,direct_system,Family history of PD,0.0044,0.0087,3
8,direct_system,MDS-UPDRS Part IV total (motor complications),0.0027,0.0095,4
9,direct_system,MoCA total (cognition),0.0012,0.0159,4


## 8. Save compact summaries

Write new or numerically equivalent CSVs and finish with integrity checks.


In [10]:
summary_validation = pd.DataFrame([
    {"check": "all upstream validations passed", "passed": bool(upstream_validation["passed"].all())},
    {"check": "two systems summarized", "passed": len(performance_table) == 2},
    {"check": "20 outer splits per target", "passed": bool(selected.groupby("target")["split_seed"].nunique().eq(20).all())},
    {"check": "five named sensitivities summarized", "passed": sensitivity_scores["sensitivity_id"].nunique() == 6},
    {"check": "26 candidate features summarized", "passed": len(statistics) == 26},
    {"check": "three full-cohort artifacts summarized", "passed": len(full_model_table) == 3},
    {"check": "interpretation covers direct and two-stage systems", "passed": set(interpretation_table["model"]) == {"direct_system", "two_stage_system"}},
])
display(summary_validation)
assert summary_validation["passed"].all()


def save_new_or_equivalent(frame, path):
    csv_text = frame.to_csv(index=False)
    if path.is_file():
        existing = pd.read_csv(path)
        current = pd.read_csv(io.StringIO(csv_text))
        pd.testing.assert_frame_equal(current, existing, check_exact=False, rtol=1e-10)
        return "already equivalent"
    path.write_text(csv_text)
    return "created"

artifacts = {
    "primary_performance.csv": performance_table,
    "paired_architecture_comparison.csv": paired_table,
    "sensitivity_summary.csv": sensitivity_table,
    "statistical_stability_summary.csv": statistical_table,
    "feature_pipeline_frequency.csv": pipeline_frequency,
    "model_family_frequency.csv": model_frequency,
    "full_cohort_model_summary.csv": full_model_table,
    "interpretation_summary.csv": interpretation_table,
    "summary_validation.csv": summary_validation,
}
save_summary = pd.DataFrame([
    {"artifact": name, "status": save_new_or_equivalent(frame, output_directory / name), "rows": len(frame)}
    for name, frame in artifacts.items()
])
display(save_summary)


,check,passed
0,all upstream validations passed,True
1,two systems summarized,True
2,20 outer splits per target,True
3,five named sensitivities summarized,True
4,26 candidate features summarized,True
5,three full-cohort artifacts summarized,True
6,interpretation covers direct and two-stage sys...,True


,artifact,status,rows
0,primary_performance.csv,created,2
1,paired_architecture_comparison.csv,created,6
2,sensitivity_summary.csv,created,12
3,statistical_stability_summary.csv,created,26
4,feature_pipeline_frequency.csv,created,22
5,model_family_frequency.csv,created,19
6,full_cohort_model_summary.csv,created,3
7,interpretation_summary.csv,created,20
8,summary_validation.csv,created,7


## Conclusion

The final 26-feature evidence is now consolidated without reopening model selection. Performance remains statistically tied between the two architectures, rare-fall recall remains the main limitation, and no sensitivity branch is promoted.
